In [3]:
import pandas as pd

In [39]:
base_annotation_dir = "/Users/sdoneva/Downloads/preclinical/08_IE_full_text/model_predictions/"
df_path_current_dataset = f"{base_annotation_dir}/full_text_combined_all_annotations_metadata.csv"
current_dataset = pd.read_csv(df_path_current_dataset)[['PMID','title', 'unique_conditions_linkbert_predictions', 'unique_interventions_linkbert_predictions','merged_mondo_label', 'merged_mondo_termid', 'merged_umls_label', 'merged_umls_termid','animal_species','animal_sex','animal_strain','animal_number','rigor_randomization','rigor_blinding','rigor_welfare','sample_size']] #'animal_age',
current_dataset.head()

/var/folders/nd/2fzvhsh510gbt9x6z5pdb1gr0000gn/T/ipykernel_19488/2379257004.py:3: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  current_dataset = pd.read_csv(df_path_current_dataset)[['PMID','title', 'unique_conditions_linkbert_predictions', 'unique_interventions_linkbert_predictions','merged_mondo_label', 'merged_mondo_termid', 'merged_umls_label', 'merged_umls_termid','animal_species','animal_sex','animal_strain','animal_number','rigor_randomization','rigor_blinding','rigor_welfare','sample_size']] #'animal_age',


,PMID,title,unique_conditions_linkbert_predictions,unique_interventions_linkbert_predictions,merged_mondo_label,merged_mondo_termid,merged_umls_label,merged_umls_termid,animal_species,animal_sex,animal_strain,animal_number,rigor_randomization,rigor_blinding,rigor_welfare,sample_size
0,1000129,Effect of morphine and naloxone on priming-ind...,audiogenic si|audiogenic seizures,naloxone|morphine,audiogenic si|audiogenic seizures,-1|MONDO:0015644,Naloxone|Morphine,C0027358|C0026549,mouse,sex-not-reported,BALB/C,not reported,randomization-not-reported,blinding-not-reported,welfare-not-reported,sample-size-not-reported
1,1000338,A morphometric investigation of the duodenal m...,vitamin d deficient,vitamin d,vitamin D deficiency,MONDO:0100471,VITAMIN D,C3714503,rat,sex-not-reported,Sprague-Dawley,not reported,randomization-present,blinding-not-reported,welfare-not-reported,sample-size-not-reported
2,10021294,Effects of IL-12 on human ovarian tumors engra...,epithelial ovarian cancer|ovarian cancer|ovari...,murine il-12|il-12|murine interleuken (il)-12,ovarian carcinoma|ovarian neoplasm,MONDO:0005140|MONDO:0021068,"Il12a protein, mouse|Interleukin 12|Edodekin a...",C1700202|C0123759|C3665495|C1121403,mouse,sex-not-reported,not reported,45.0,randomization-not-reported,blinding-not-reported,welfare-not-reported,sample-size-not-reported
3,10021348,Mechanisms of GDF-5 action during skeletal dev...,chondrody,gdf 5|gdf-5,obsolete cartilage disease,MONDO:0005569,Growth Differentiation Factor 5,C0253373,mouse,sex-not-reported,not reported,not reported,randomization-not-reported,blinding-not-reported,welfare-not-reported,sample-size-not-reported
4,10022166,Prenatal vitamin E treatment improves lung gro...,congenital diaphragmatic hernia,vitamin e,congenital diaphragmatic hernia,MONDO:0005711,Vitamin E,C0042874,rat,sex-female,Sprague-Dawley,not reported,randomization-not-reported,blinding-not-reported,welfare-not-reported,sample-size-not-reported


In [40]:
import pandas as pd

multiple_assignments = current_dataset[
    current_dataset["merged_mondo_label"].fillna("").str.contains(r"\|")
    | current_dataset["merged_umls_label"].fillna("").str.contains(r"\|")
].copy()

sampled = (
    multiple_assignments
    .sample(n=min(50, len(multiple_assignments)), random_state=42)
    [
        [
            "PMID",
            "title",
            "unique_conditions_linkbert_predictions",
            "merged_mondo_label",
            "merged_mondo_termid",
            "unique_interventions_linkbert_predictions",
            "merged_umls_label",
            "merged_umls_termid",
        ]
    ]
    .copy()
)


def split_values(value):
    """Split pipe-separated values and remove surrounding whitespace."""
    if pd.isna(value) or str(value).strip() == "":
        return [pd.NA]

    return [
        item.strip()
        for item in str(value).split("|")
    ]


def make_pairs(row, label_col, id_col):
    """Pair each normalized label with its corresponding term ID."""
    labels = split_values(row[label_col])
    term_ids = split_values(row[id_col])

    if len(labels) != len(term_ids):
        raise ValueError(
            f"Different numbers of labels and IDs for PMID {row['PMID']}: "
            f"{label_col}={len(labels)}, {id_col}={len(term_ids)}"
        )

    return list(zip(labels, term_ids))


# Keep normalized labels and IDs paired
sampled["mondo_pair"] = sampled.apply(
    make_pairs,
    axis=1,
    label_col="merged_mondo_label",
    id_col="merged_mondo_termid",
)

sampled["umls_pair"] = sampled.apply(
    make_pairs,
    axis=1,
    label_col="merged_umls_label",
    id_col="merged_umls_termid",
)

# Create one row per possible disease–drug combination
sampled_expanded = (
    sampled[
        [
            "PMID",
            "title",
            "unique_conditions_linkbert_predictions",
            "unique_interventions_linkbert_predictions",
            "mondo_pair",
            "umls_pair",
        ]
    ]
    .explode("mondo_pair")
    .explode("umls_pair")
    .reset_index(drop=True)
)

# Separate paired disease labels and IDs
sampled_expanded[
    ["merged_mondo_label", "merged_mondo_termid"]
] = pd.DataFrame(
    sampled_expanded["mondo_pair"].tolist(),
    index=sampled_expanded.index,
)

# Separate paired drug labels and IDs
sampled_expanded[
    ["merged_umls_label", "merged_umls_termid"]
] = pd.DataFrame(
    sampled_expanded["umls_pair"].tolist(),
    index=sampled_expanded.index,
)

sampled_expanded = (
    sampled_expanded
    .drop(columns=["mondo_pair", "umls_pair"])
    [
        [
            "PMID",
            "title",
            "unique_conditions_linkbert_predictions",
            "merged_mondo_label",
            "merged_mondo_termid",
            "unique_interventions_linkbert_predictions",
            "merged_umls_label",
            "merged_umls_termid",
        ]
    ]
)

sampled_expanded.head()

,PMID,title,unique_conditions_linkbert_predictions,merged_mondo_label,merged_mondo_termid,unique_interventions_linkbert_predictions,merged_umls_label,merged_umls_termid
0,2056397,Enhanced resection and improved survival in mu...,neuroblastoma,neuroblastoma,MONDO:0005072,retinyl palmitate,Vitamin A palmitate,C0073115
1,2056397,Enhanced resection and improved survival in mu...,neuroblastoma,neuroblastoma,MONDO:0005072,retinyl palmitate,Retinyl Esters,C0807756
2,36988254,Protective effects of cordycepin pretreatment ...,liver ischemia/reperfusion injury|ischemia/rep...,ischemia reperfusion injury,MONDO:0005203,cordycepin,Cordycepin,C0056331
3,36988254,Protective effects of cordycepin pretreatment ...,liver ischemia/reperfusion injury|ischemia/rep...,ir-induced liver injury,-1,cordycepin,Cordycepin,C0056331
4,36359010,Double-Edged Sword Effect of Pyroptosis: The R...,apical periodontitis,periapical periodontitis,MONDO:0004508,caspase-1 /-4 /-5 inhibitor ac-ftdl-cmk|caspas...,caspase-1 /-4 /-5 inhibitor ac-ftdl-cmk,-1


In [43]:
sampled_expanded.to_csv("outputs/sampled_50_validation_drug_disease.csv", index=False)